In [1]:
import pymysql
import pandas as pd
import json
import os
import sys
import requests
sys.path.append("/home/trade_core")
from postgres_client import InitDBClient
from psycopg2 import extras
import time

In [2]:
with open('/home/trade_core/trade_core_config.json') as config_file:
    config = json.load(config_file)

node_db_name = config['node_settings'][config['node']]['db_settings']
node_db_setting_dict = config['database_setting'][node_db_name]

In [3]:
db_client = InitDBClient(**{**node_db_setting_dict, 'database': 'trade_core'})

InitDBClient|SCHEMA: trade_core already exists.


In [4]:
db_client.create_all_tables()

InitDBClient|TABLE: trade_config already exists.
InitDBClient|TABLE: trade already exists.
InitDBClient|TABLE: trade_log created.
InitDBClient|TABLE: repeat_trade already exists.
InitDBClient|TABLE: exchange_api_key already exists.
InitDBClient|TABLE: order_history created.
InitDBClient|TABLE: trade_history created.


In [8]:
def is_table_empty(conn, table_name):
    with conn.cursor() as cur:
        cur.execute(f"SELECT COUNT(*) FROM {table_name}")
        count = cur.fetchone()[0]
        return count == 0

def get_column_names(conn, table_name):
    with conn.cursor() as cur:
        cur.execute("""
            SELECT column_name 
            FROM information_schema.columns 
            WHERE table_schema = 'public' AND table_name = %s
            ORDER BY ordinal_position;
        """, (table_name,))
        return [row[0] for row in cur.fetchall()]

In [8]:
high_break_trigger_uuid_list = ['57f5df1d-95f9-49eb-b7e4-c15042c9b59f', '00dc872d-b31d-412d-94e5-9de9026f4a48', '00dc872d-b31d-412d-94e5-9de9026f4a48']

conn = db_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)

curr.execute("SELECT * FROM trade WHERE uuid IN %s", (tuple(high_break_trigger_uuid_list),))
result = curr.fetchall()

db_client.pool.putconn(conn)

In [6]:
conn = db_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)


A = 'trade'
B = "order_history"
C = "trade_history"


sql = f"""
    SELECT 
    {A}.*,
    {B}.order_id AS b_order_id, {B}.order_type AS b_order_type, {B}.market_code AS b_market_code, 
    {B}.symbol AS b_symbol, {B}.quote_asset AS b_quote_asset, {B}.side AS b_side, 
    {B}.price AS b_price, {B}.qty AS b_qty, {B}.fee AS b_fee, {B}.remark AS b_remark,
    {C}.uuid AS c_uuid, {C}.registered_datetime AS c_registered_datetime, {C}.trade_side AS c_trade_side, 
    {C}.base_asset AS c_base_asset, {C}.target_order_id AS c_target_order_id, 
    {C}.origin_order_id AS c_origin_order_id, {C}.dollar AS c_dollar, {C}.remark AS c_remark
    FROM 
        {A}
        INNER JOIN {B} ON {A}.trade_config_uuid = {B}.trade_config_uuid AND {A}.uuid = {B}.trade_uuid
        INNER JOIN {C} ON {B}.trade_config_uuid = {C}.trade_config_uuid AND {B}.trade_uuid = {C}.trade_uuid

"""
curr = conn.cursor(cursor_factory=extras.RealDictCursor)
curr.execute(sql)
result = curr.fetchall()

db_client.pool.putconn(conn)

In [32]:
start = time.time()
for i in range(1000):
    conn = db_client.pool.getconn()
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    sql = """
        SELECT * FROM trade"""
    curr.execute(sql)
    result = curr.fetchall()
    # print(result)
    db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

Time: 0.18970322608947754


In [60]:
start = time.time()
conn = db_client.pool.getconn()
for i in range(1000):
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    sql = """
        SELECT * FROM trade"""
    curr.execute(sql)
    result = curr.fetchall()
    # print(result)
db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

Time: 0.14306998252868652


In [75]:
start = time.time()
for i in range(1000):
    conn = db_client.pool.getconn()
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    sql = """
        SELECT trade.*, trade_config.uuid, trade_config, trade_config.service_datetime_end, trade_config.telegram_id, trade_config.target_market_code, trade_config.origin_market_code
        FROM trade
        JOIN trade_config ON trade.trade_config_uuid = trade_config.uuid WHERE trade_config.target_market_code='UPBIT_SPOT/KRW' AND trade_config.origin_market_code='BINANCE_USD_M/USDT' AND trade_config.service_datetime_end <= %s"""
    val = (pd.Timestamp.now(),)
    curr.execute(sql, val)
    result = curr.fetchall()
    # print(result)
    db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

Time: 0.3250555992126465


In [79]:
start = time.time()
conn = db_client.pool.getconn()
for i in range(1000):
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    sql = """
        SELECT trade.*, trade_config.uuid, trade_config, trade_config.service_datetime_end, trade_config.telegram_id, trade_config.target_market_code, trade_config.origin_market_code
        FROM trade
        JOIN trade_config ON trade.trade_config_uuid = trade_config.uuid WHERE trade_config.target_market_code='UPBIT_SPOT/KRW' AND trade_config.origin_market_code='BINANCE_USD_M/USDT' AND trade_config.service_datetime_end <= %s"""
    val = (pd.Timestamp.now(),)
    curr.execute(sql, val)
    result = curr.fetchall()
    # print(result)
db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

Time: 0.29623842239379883


In [81]:
start = time.time()
conn = db_client.pool.getconn()
for i in range(100):
    curr = conn.cursor(cursor_factory=extras.RealDictCursor)
    sql = """
        SELECT trade.*, trade_config.uuid, trade_config, trade_config.service_datetime_end, trade_config.telegram_id, trade_config.target_market_code, trade_config.origin_market_code
        FROM trade
        JOIN trade_config ON trade.trade_config_uuid = trade_config.uuid"""
    val = (pd.Timestamp.now(),)
    curr.execute(sql)
    result = curr.fetchall()
    # print(result)
db_client.pool.putconn(conn)
print(f"Time: {time.time() - start}")

Time: 0.033699989318847656


In [71]:
result

[RealDictRow([('id', 12),
              ('uuid', '57f5df1d-95f9-49eb-b7e4-c15042c9b59f'),
              ('trade_config_uuid', '57f5df1d-95f9-49eb-b7e4-c15042c9b59f'),
              ('registered_datetime',
               datetime.datetime(2024, 2, 2, 8, 59, 21, 317110)),
              ('last_updated_datetime',
               datetime.datetime(2024, 2, 2, 8, 59, 21, 317123)),
              ('base_asset', 'BTC'),
              ('usdt_conversion', False),
              ('low', Decimal('3.000')),
              ('high', Decimal('4.000')),
              ('trigger_switch', None),
              ('trade_switch', None),
              ('trade_capital', None),
              ('enter_target_market_order_id', None),
              ('enter_origin_market_order_id', None),
              ('exit_target_market_order_id', None),
              ('exit_origin_market_order_id', None),
              ('status', None),
              ('remark', None),
              ('trade_config',
               '(2,57f5df1d-95f9-49

In [35]:
0.36/1000

0.00035999999999999997

In [ ]:
# First check whether the table is empty
conn = db_client.pool.getconn()
curr = conn.cursor(cursor_factory=extras.RealDictCursor)
if is_table_empty(conn, 'trade_config'):
    # # Get the column names
    # column_names = self.get_column_names(conn, table_name)
    # # Create empty dataframe
    # self.exchange_config_df = pd.DataFrame(columns=column_names)
    pass
else:
    curr.execute(f"SELECT * FROM {table_name}")
    exchange_config_df = pd.DataFrame(curr.fetchall())
    target_market_code_unique = exchange_config_df['target_market_code'].unique()
    origin_market_code_unique = exchange_config_df['origin_market_code'].unique()
    for target_market_code in target_market_code_unique:
        for origin_market_code in origin_market_code_unique:
            market_combi_code = f"{target_market_code}:{origin_market_code}"
            self.exchange_config_df_dict[market_combi_code] = exchange_config_df[(exchange_config_df['target_market_code'] == target_market_code) & (exchange_config_df['origin_market_code'] == origin_market_code)]
self.postgres_client.pool.putconn(conn)
if self.exchange_config_df_dict_initiated is False:
    self.exchange_config_df_dict_initiated = True

In [5]:
response = requests.get("https://arbicrypto.net/api/users/users/")